<a href="https://colab.research.google.com/github/neelameghana192311002/CSA6301---Threat-Intelligence-and-Network-Security/blob/main/UNIT4LAB/Exercise_4_Inline_IPS_Blocking_Simulator_with_Alert_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
IDS_RULES = [
    {"type": "keyword", "value": "union select username,password from users"}
]
# Placeholder for IDS_RULES, you might want to load actual rules here.

In [8]:
def scan_packet(packet, rules):
    # This placeholder function is updated to simulate alert detection.
    # In a real scenario, this would contain logic to scan the packet
    # against the given rules and return any alerts.
    alerts = []
    for rule in rules:
        if rule["type"] == "keyword" and rule["value"] in packet.get("payload", ""):
            alerts.append({"rule": rule["value"], "severity": "high"})
    print(f"Scanning packet: {packet} with rules: {rules}. Alerts: {alerts}")
    return alerts

In [9]:
def inspect_and_decide(packet, rules=IDS_RULES, whitelist=None):
    """
    Like scan_packet, but decides whether to BLOCK (untrusted source)
    or ALLOW-WITH-LOG (source is on the tuning whitelist, e.g. a known
    partner IP that reliably triggers a benign false positive).
    """
    whitelist = whitelist or set()

    alerts = scan_packet(packet, rules)

    if not alerts:
        return {
            "action": "allow",
            "alerts": []
        }

    if packet.get("src_ip") in whitelist:
        return {
            "action": "allow",
            "alerts": alerts,
            "note": "source whitelisted, alert suppressed from blocking"
        }

    return {
        "action": "block",
        "alerts": alerts
    }

In [10]:
def test_experiment4():
    attack_packet = {
        "proto": "tcp",
        "dst_port": 80,
        "payload": "union select username,password from users",
        "src_ip": "203.0.113.50",
    }

    result = inspect_and_decide(attack_packet, IDS_RULES)
    assert result["action"] == "block"

    # Identical signature match, but from a whitelisted partner IP
    partner_packet = dict(
        attack_packet,
        src_ip="198.51.100.10"
    )

    result2 = inspect_and_decide(
        partner_packet,
        IDS_RULES,
        whitelist={"198.51.100.10"}
    )

    assert result2["action"] == "allow"
    assert "note" in result2

    print("All test cases passed.")


test_experiment4()

Scanning packet: {'proto': 'tcp', 'dst_port': 80, 'payload': 'union select username,password from users', 'src_ip': '203.0.113.50'} with rules: [{'type': 'keyword', 'value': 'union select username,password from users'}]. Alerts: [{'rule': 'union select username,password from users', 'severity': 'high'}]
Scanning packet: {'proto': 'tcp', 'dst_port': 80, 'payload': 'union select username,password from users', 'src_ip': '198.51.100.10'} with rules: [{'type': 'keyword', 'value': 'union select username,password from users'}]. Alerts: [{'rule': 'union select username,password from users', 'severity': 'high'}]
All test cases passed.
